# Zero copy

`mtl5.vector(a)` and `mtl5.matrix(A)` don't copy a NumPy array. They lay an MTL5
container over its memory. MTL5 then reads that memory in place, and the
in-place operations write to it, so data that already lives in a NumPy array (or
in a PyTorch tensor, or in a buffer a C library filled) crosses into MTL5 at no
cost.

This notebook works through the rules that make that safe. Then it applies
them to the two settings of the C++ examples: an ODE integrator's callback that
writes the integrator's own buffer, and frames processed in the memory they
arrived in.

**Uses:**
- `mtl5.vector`, `mtl5.matrix`, `.is_view`, `.to_numpy()`, `.copy()`,
  `mtl5.vector_copy`;
- the in-place operations `mtl5.symv` and `mtl5.trsv`;
- `mtl5.array.asarray`, for strided views.

**From C++:** [`zero_copy/zero_copy_rules.cpp`](https://github.com/stillwater-sc/mtl5/blob/v5.25.0/examples/zero_copy/zero_copy_rules.cpp), [`zero_copy/ode_rhs_callback.cpp`](https://github.com/stillwater-sc/mtl5/blob/v5.25.0/examples/zero_copy/ode_rhs_callback.cpp) and [`zero_copy/shared_frame_pipeline.cpp`](https://github.com/stillwater-sc/mtl5/blob/v5.25.0/examples/zero_copy/shared_frame_pipeline.cpp).

In [ ]:
import gc
import queue
import threading
import weakref

import matplotlib.pyplot as plt
import numpy as np

import mtl5
import mtl5.sparse

# Chart styling: recessive axes and grid, ink for text, validated categorical hues.
SURFACE = "#fcfcfb"
INK, INK_2 = "#0b0b0b", "#52514e"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": "#d8d7d2",
        "axes.labelcolor": INK_2,
        "axes.titlecolor": INK,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "grid.color": "#e8e7e2",
        "grid.linewidth": 0.8,
        "text.color": INK,
        "xtick.color": INK_2,
        "ytick.color": INK_2,
        "font.size": 10,
        "figure.dpi": 120,
    }
)

checks = {}  # each rule's claim, verified as it is shown

## 1. Writes go through, both ways

A view and its array are one block of memory. A write through either is seen
by the other, and `.to_numpy()` (or `np.asarray`) on a view returns the same
memory again.

In [ ]:
a = np.zeros(4)
v = mtl5.vector(a)
v[0] = 1.0  # through MTL5
a[1] = 2.0  # through NumPy
print(v.is_view, a, np.asarray(v))
checks["write-through"] = v.is_view and a[0] == 1.0 and np.asarray(v)[1] == 2.0
checks["round trip"] = np.shares_memory(np.asarray(v), a) and np.shares_memory(v.to_numpy(), a)

## 2. What can't be viewed is refused, not copied

An MTL5 vector or dense matrix is contiguous: it holds a pointer and a length,
or a pointer and a row-major shape. An array MTL5 can't describe that way is
refused with a `TypeError`. A silent copy would break rule 1 without a word.

The refused cases:
- **Strided or column-major:** for example `a[::2]`, or a Fortran-order matrix.
- **Read-only:** a view can be written, so it needs a writable array.
- **No matching type:** an element type mtl5 has no container for, such as
  float16 or bool.

In [ ]:
def refused(make):
    try:
        make()
    except TypeError:
        return True
    return False


ro = np.arange(3.0)
ro.flags.writeable = False
cases = {
    "strided a[::2]": lambda: mtl5.vector(np.arange(6.0)[::2]),
    "Fortran-order matrix": lambda: mtl5.matrix(np.asfortranarray(np.ones((2, 3)))),
    "read-only": lambda: mtl5.vector(ro),
    "float16": lambda: mtl5.vector(np.zeros(3, dtype=np.float16)),
    "bool": lambda: mtl5.vector(np.zeros(3, dtype=bool)),
}
for name, make in cases.items():
    print(f"{name:>22}: {'refused' if refused(make) else 'accepted'}")
checks["refusals"] = all(refused(make) for make in cases.values())

## 3. Copies are explicit, and results are new

To copy, ask for it: `.copy()` on a view, or `mtl5.vector_copy(a)` and
`mtl5.matrix_copy(A)` on an array. A copy owns its memory (`is_view` is
`False`).

An operation that returns a value, such as `mtl5.matvec` or `mtl5.solve`,
reads its inputs in place and returns a new, owning result. To write a
result into memory you already have, use the in-place operations, which take
the output as an argument:

- `mtl5.symv(alpha, A, x, beta, y)`: $y \leftarrow \alpha A x + \beta y$;
- `mtl5.trsv(A, x)`: $x \leftarrow A^{-1} x$, for triangular $A$;
- `ger`, `trmv`, `trmm`, `trsm`, `symm`, `syrk` and `syr2k`: the rest of the
  BLAS-style in-place family.

In [ ]:
a = np.arange(1.0, 4.0)
c = mtl5.vector(a).copy()
c[0] = 100.0
r = mtl5.matvec(mtl5.matrix(np.eye(3)), mtl5.vector(a))
print("copy:", c.is_view, a, "  matvec result:", r.is_view)

A = mtl5.generators.lehmer(3).to_numpy()
y = np.zeros(3)
mtl5.symv(1.0, mtl5.matrix(A), mtl5.vector(a), 0.0, mtl5.vector(y))  # writes y
b = np.triu(A) @ a
mtl5.trsv(mtl5.matrix(np.triu(A)), mtl5.vector(b))  # overwrites b with the solution
print("symv wrote y:", y, "  trsv left b =", b)
checks["copies"] = not c.is_view and a[0] == 1.0 and not r.is_view
checks["in place"] = np.allclose(y, A @ a) and np.allclose(b, a)

## 4. Aliasing: an in-place operation may overwrite its own input

The C++ rules note that MTL5 evaluates an expression that reads its own target
through a temporary. The in-place operations here behave the same way:
`symv` with `y` and `x` over the *same* buffer computes $y \leftarrow A y$
correctly. A concurrent write from another thread is a different matter
(rule 7).

In [ ]:
x = np.arange(1.0, 4.0)
expected = A @ x
vx = mtl5.vector(x)
mtl5.symv(1.0, mtl5.matrix(A), vx, 0.0, vx)  # y and x alias
print(x, expected)
checks["aliasing"] = np.array_equal(x, expected)

## 5. Strided data: `mtl5.array`

Image and DMA buffers often pad each row to a *pitch*. The visible image is
then a strided slice of the buffer, which rule 2 keeps out of `mtl5.matrix`.
`mtl5.array.asarray` takes any strides: its N-dimensional array is a view with
a shape and strides, like NumPy's. Slicing it gives views too.

In [ ]:
x_pad = -1.0
buffer = np.array([[1, 2, 3, x_pad], [4, 5, 6, x_pad]])  # a 2 x 3 image, pitch 4
image = buffer[:, :3]
print("mtl5.matrix:", "refused" if refused(lambda: mtl5.matrix(image)) else "accepted")
nd = mtl5.array.asarray(image)
nd[1, 1] = 50.0
print("strides", nd.strides, "is_view", nd.is_view, "sum", nd.sum())
print(buffer)
checks["strided"] = nd.is_view and buffer[1, 1] == 50.0 and nd.sum() == 1 + 2 + 3 + 4 + 50 + 6

## 6. Lifetime: a view keeps its array alive

A view holds a reference to the array it borrows from. So the array outlives
every view of it, even when the program drops its own name for it. This is
the one rule where Python is safer than C++: there, the buffer's lifetime is
the caller's responsibility.

In [ ]:
a = np.arange(4.0)
alive = weakref.ref(a)
v = mtl5.vector(a)
del a
gc.collect()
kept = alive() is not None and np.array_equal(np.asarray(v), [0, 1, 2, 3])
del v
gc.collect()
print("kept alive by the view:", kept, "  freed with it:", alive() is None)
checks["lifetime"] = kept and alive() is None

## 7. Synchronization is yours

MTL5's kernels release the GIL, so other Python threads run while they do. MTL5
takes no lock on the memory it views. A buffer that two threads share needs a
handoff, as in the frame pipeline below.

## An ODE right-hand side that writes the integrator's buffer

A library that owns the state often calls *your* function with its own
buffers: an ODE integrator, an optimizer, a C library through `ctypes`. The
integrator here is a classical RK4 written like a C library. It owns $y$ and
the stage buffers, and it calls `rhs(t, y, dydt)`, which must fill `dydt`.

The problem is the 1D heat equation $u_t = u_{xx}$ on $n = 200$ interior points,
semi-discretized to $\dot y = L y$. The right-hand side wraps the two buffers
and lets `symv` write $L y$ straight into `dydt`. Nothing is copied in or out,
and the callback checks that on every call.

$\sin(\pi x)$ is an eigenvector of the discrete Laplacian, with eigenvalue
$\lambda_1$, so the exact solution of the semi-discrete system is
$e^{\lambda_1 t}$ times the initial state.

In [ ]:
def rk4_integrate(rhs, y, t0, t1, steps):
    # A fixed-step RK4 that owns its buffers and hands them to rhs, like a C
    # integrator. It updates y in place.
    k1, k2, k3, k4, tmp = (np.empty_like(y) for _ in range(5))
    dt = (t1 - t0) / steps
    t = t0
    for _ in range(steps):
        rhs(t, y, k1)
        np.multiply(k1, 0.5 * dt, out=tmp)
        tmp += y
        rhs(t + 0.5 * dt, tmp, k2)
        np.multiply(k2, 0.5 * dt, out=tmp)
        tmp += y
        rhs(t + 0.5 * dt, tmp, k3)
        np.multiply(k3, dt, out=tmp)
        tmp += y
        rhs(t + dt, tmp, k4)
        y += dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
        t += dt


n, t1, steps = 200, 0.02, 2000
h = 1.0 / (n + 1)
L = mtl5.matrix(mtl5.sparse.to_scipy(mtl5.generators.laplacian_1d(n)).toarray() * (-1.0 / h**2))
stats = {"calls": 0, "in place": 0}


def heat_rhs(t, y, dydt):
    Y, dY = mtl5.vector(y), mtl5.vector(dydt)  # views of the integrator's buffers
    mtl5.symv(1.0, L, Y, 0.0, dY)  # dydt = L y, written in place
    stats["calls"] += 1
    stats["in place"] += Y.is_view and dY.is_view and np.shares_memory(np.asarray(dY), dydt)


xs = h * np.arange(1, n + 1)
y = np.sin(np.pi * xs)
rk4_integrate(heat_rhs, y, 0.0, t1, steps)
lam1 = -4 / h**2 * np.sin(np.pi * h / 2) ** 2
exact = np.exp(lam1 * t1) * np.sin(np.pi * xs)
rel_err = np.abs(y - exact).max() / np.abs(exact).max()
print(f"{stats['calls']} rhs calls, {stats['in place']} in place; relative max error {rel_err:.2e}")

fig, ax = plt.subplots(figsize=(6.4, 3.2))
ax.plot(xs, np.sin(np.pi * xs), color=INK_2, lw=1, ls="--", label="t = 0")
ax.plot(xs, y, color=SERIES[0], lw=2, label=f"RK4 through the callback, t = {t1}")
ax.plot(xs[::12], exact[::12], "o", color=SERIES[1], ms=4, label="exact")
ax.set_xlabel("x")
ax.set_ylabel("u")
ax.legend(fontsize=8)
ax.grid(True)
fig.tight_layout()
plt.show()
checks["ode"] = rel_err < 1e-8 and stats["in place"] == stats["calls"] == 4 * steps

`generators.laplacian_1d` is tridiag(−1, 2, −1), sparse, so $L$ is minus it
over $h^2$. The C++ example keeps $L$ sparse. Here it is made dense, because
`symv` is the in-place product mtl5 binds, and at $n = 200$ a dense product
costs microseconds. The sparse matrices' `matvec` returns a new array.

## Frames processed where they arrived

The C++ pipeline has a producer thread that fills frames in a shared region of
two slots, and a processing thread that corrects each frame in place and
hands the slot back. The handoff here is two queues: the producer puts a
filled slot's number on one, and the processor returns it on the other once
done, so the two threads never touch the same slot at once (rule 7).

For each frame, the processor makes two corrections:

- **Dark frame and gain**, over the whole frame: through the view's
  `.to_numpy()`, which is the slot's own memory.
- **Background flattening**, per 32 × 32 tile: a tile's rows aren't contiguous,
  so it is a `mtl5.array` slice. MTL5 computes its mean in place, and NumPy
  subtracts the mean from the tile in place.

The consumer checks each frame against values computed independently.

In [ ]:
H, W, TILE, SLOTS, FRAMES, GAIN = 96, 128, 32, 2, 24, np.float32(2.5)
rr, cc = np.mgrid[0:H, 0:W]
dark = (100 + (rr * 7 + cc * 13) % 17).astype(np.float32)
tile_id = (rr // TILE) * (W // TILE) + cc // TILE


def scene(k):
    return 50 + 40 * np.sin(0.1 * rr + 0.07 * cc + 0.3 * k)


def background(k):
    return 10 + 3 * tile_id + k


region = np.zeros((SLOTS, H, W), dtype=np.float32)  # one allocation, two slots
filled, empty = queue.Queue(), queue.Queue()
for s in range(SLOTS):
    empty.put(s)


WAIT = 30  # seconds: a worker whose partner has died stops instead of hanging
errors = []


def worker(body):
    # Run body; record an exception for the main thread to raise.
    def run():
        try:
            body()
        except BaseException as e:  # noqa: BLE001 -- re-raised after join
            errors.append(e)

    return threading.Thread(target=run)


def producer():
    for k in range(FRAMES):
        s = empty.get(timeout=WAIT)  # wait for a free slot
        region[s] = dark + ((scene(k) + background(k)) / GAIN).astype(np.float32)
        filled.put((s, k))


report = {"frames": 0, "in place": 0, "max error": 0.0}


def process(slot):
    F = mtl5.matrix(slot)  # the whole frame, a view of the slot
    f = F.to_numpy()
    f -= dark
    f *= GAIN
    in_place = F.is_view and np.shares_memory(f, region)
    nd = mtl5.array.asarray(slot)
    for r0 in range(0, H, TILE):
        for c0 in range(0, W, TILE):
            tile = nd[r0 : r0 + TILE, c0 : c0 + TILE]  # strided view
            mean = tile.mean()  # computed by MTL5, in place
            t = tile.to_numpy()
            t -= np.float32(mean)
            in_place = in_place and tile.is_view and np.shares_memory(t, region)
    return in_place


def processor():
    for _ in range(FRAMES):
        s, k = filled.get(timeout=WAIT)
        report["in place"] += process(region[s])
        expected = scene(k)
        for t in range(tile_id.max() + 1):  # scene minus its tile mean
            expected[tile_id == t] -= expected[tile_id == t].mean()
        error = float(np.abs(region[s] - expected).max())
        # max() would drop a NaN, so a non-finite error is recorded as inf.
        report["max error"] = max(report["max error"], error if np.isfinite(error) else np.inf)
        report["frames"] += 1
        empty.put(s)  # hand the slot back


threads = [worker(producer), worker(processor)]
for th in threads:
    th.start()
for th in threads:
    th.join()
if errors:
    raise errors[0]
print(
    f"{report['frames']} frames, {report['in place']} processed in place, "
    f"max error {report['max error']:.2e}"
)

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].imshow(dark + ((scene(FRAMES - 1) + background(FRAMES - 1)) / GAIN), cmap="gray")
axes[0].set_title("raw frame (last)", fontsize=10)
axes[1].imshow(region[(FRAMES - 1) % SLOTS], cmap="gray")
axes[1].set_title("after correction, in its slot", fontsize=10)
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
fig.tight_layout()
plt.show()
checks["frames"] = report["frames"] == FRAMES == report["in place"] and report["max error"] < 1e-3

## Check

Each rule's claim was recorded above, as it was shown.

In [ ]:
for name, ok in checks.items():
    print(f"{name:>14}: {'ok' if ok else 'FAILED'}")
assert all(checks.values()), [name for name, ok in checks.items() if not ok]
print("all checks passed")

## From C++

- **[`zero_copy/zero_copy_rules.cpp`](https://github.com/stillwater-sc/mtl5/blob/v5.25.0/examples/zero_copy/zero_copy_rules.cpp)** walks through the C++ rules. Some of them don't arise
  in Python:
  - Assignment and size: Python's `=` rebinds a name, and never assigns into a
    view.
  - Resizing.
  - Copy semantics: in C++, copying a vector that is a view makes an owning
    copy.

  The rules that carry over are write-through, layout, aliasing, lifetime and
  synchronization. Python adds the refusals, and makes lifetime automatic.
- **[`zero_copy/ode_rhs_callback.cpp`](https://github.com/stillwater-sc/mtl5/blob/v5.25.0/examples/zero_copy/ode_rhs_callback.cpp)**: the same RK4, heat equation and check. In C++ the
  right-hand side is a sparse `mult` into a vector over `dydt`. The example also
  times copy-in/copy-out against zero copy; that comparison is left out here.
- **[`zero_copy/shared_frame_pipeline.cpp`](https://github.com/stillwater-sc/mtl5/blob/v5.25.0/examples/zero_copy/shared_frame_pipeline.cpp)**: the same frames, scene, tiles and check. The C++
  threads hand slots over with a mutex and a condition variable; here, two
  queues. In C++ the correction is an MTL5 expression and the tile update an
  `ndarray` operation. Here NumPy does the element-wise work on the same memory,
  and MTL5 the tile means.